# 06 DPO 从偏好数据到 DPOTrainer：为什么指标好也可能不接受

DPO 的数据不是标准答案，而是偏好对：同一个 prompt 下，chosen 比 rejected 更好。

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, textwrap, re

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = Path(r"D:/coding/qwen lorar sft/qwen-local-lora-sft-dpo")

print("项目根目录:", PROJECT_ROOT)
print("学习原则: 本 notebook 默认只读项目文件；所有真实推理/训练开关默认 False。")

def read_text(rel):
    return (PROJECT_ROOT / rel).read_text(encoding="utf-8", errors="replace")

def show_file(rel, start=1, end=None):
    lines = read_text(rel).splitlines()
    if end is None:
        end = len(lines)
    print(f"--- {rel}:{start}-{end} ---")
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:03d}: {lines[i-1]}")

def find_lines(rel, keyword, context=4):
    lines = read_text(rel).splitlines()
    hits = []
    for idx, line in enumerate(lines, start=1):
        if keyword in line:
            hits.append(idx)
    if not hits:
        print("没有找到:", keyword)
        return
    for hit in hits:
        start = max(1, hit - context)
        end = min(len(lines), hit + context)
        show_file(rel, start, end)
        print()

def read_jsonl(rel, n=3):
    rows = []
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
            if len(rows) >= n:
                break
    return rows

def count_jsonl(rel):
    with (PROJECT_ROOT / rel).open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

## 1. 打开一条 DPO 数据

In [ ]:
for rel in ["data/processed/dpo_tiny_train.jsonl", "data/processed/dpo_larger_v6_train.jsonl"]:
    print(rel, "rows:", count_jsonl(rel))
row = read_jsonl("data/processed/dpo_tiny_train.jsonl", n=1)[0]
print(json.dumps(row, ensure_ascii=False, indent=2)[:3000])

## 2. 代码怎么校验 prompt/chosen/rejected？

In [ ]:
find_lines("scripts/train_dpo.py", "def read_dpo_rows", context=20)
find_lines("scripts/train_dpo.py", "chosen", context=8)

DPO 训练需要 prompt、chosen、rejected 三列。如果任意一项空了，脚本会报错，因为这条偏好样本不完整。

## 3. DPO 的 policy model 和 reference model

In [ ]:
find_lines("scripts/train_dpo.py", "def load_policy_model", context=18)
find_lines("scripts/train_dpo.py", "def load_reference_model", context=18)

policy model 是正在训练的模型；reference model 是冻结参考模型，用来约束 policy 不要偏离太离谱。本项目 v6 使用 separate frozen reference model。

## 4. DPOTrainer 在哪里？

In [ ]:
find_lines("scripts/train_dpo.py", "DPOConfig", context=20)
find_lines("scripts/train_dpo.py", "DPOTrainer", context=16)

这些参数要会解释：`beta` 是偏好优化强度相关参数；`max_length` / `max_prompt_length` 控制显存；`per_device_train_batch_size=1` 是 8GB 显存保守设置；`gradient_accumulation_steps` 用小 batch 累积。

## 5. 为什么 DPO 没成为最终推荐？

DPO v6/v7/v8 的 preference 指标可以很好看，但固定 prompt 7 仍没有稳定通过。这说明 preference accuracy 高不等于真实行为在目标 prompt 上稳定正确。

## 6. 你要能讲出的底层句子

> DPO 使用 prompt/chosen/rejected 偏好对训练 policy adapter，并可用 frozen reference model 约束更新。但本项目最终没有只看 preference accuracy，因为固定 prompt 行为门禁发现 DPO 仍无法稳定修复 prompt 7。